# M20 — Understand Optimization Experimentally

**Objective:** study learning dynamics by manipulating optimization behavior. M19 established what a gradient means; here the objective and gradient stay fixed while learning rate, stochasticity, momentum, and Adam change the path.

## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running** each action cell and timestamp the prediction in your own evidence log. Do not type learner answers into this source notebook. The lab is deterministic where stochasticity is controlled, CPU-only, offline, secret-free, and paid-API-free.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M20" / "optimization_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M20.optimization_core import (
    DEFAULT_CURVATURES,
    DEFAULT_INITIALIZATION,
    component_noise_mean,
    coordinate_sign_changes,
    diagnose_dynamics,
    loss_history,
    quadratic_gradient,
    quadratic_loss,
    run_learning_rate_sweep,
    run_optimizer,
)

print("repository root:", ROOT)
print("objective curvatures:", DEFAULT_CURVATURES)
print("initialization:", DEFAULT_INITIALIZATION)

## M19 boundary: keep the gradient mechanism fixed

The objective is `L(theta) = 0.5 * (theta_0² + 10 * theta_1²)`, with gradient `(theta_0, 10 * theta_1)`. Unequal curvature makes learning-rate stability visible. M20 manipulates dynamics after the gradient; it does not change derivative rules to rescue a run.

In [ ]:
initial_loss = quadratic_loss(DEFAULT_INITIALIZATION)
initial_gradient = quadratic_gradient(DEFAULT_INITIALIZATION)
print("initial loss:", initial_loss)
print("initial gradient:", initial_gradient)
assert initial_loss == 88.0
assert initial_gradient == (4.0, 40.0)

## Predict before running — learning-rate regimes

For exact gradient descent at rates `1e-5`, `0.05`, `0.19`, and `0.21`, sketch each loss curve and high-curvature parameter path. Predict which run stagnates, converges smoothly, oscillates while converging, or diverges. Include one numeric or sign-pattern claim that could falsify each prediction.

In [ ]:
learning_rates = (1.0e-5, 0.05, 0.19, 0.21)
rate_traces = run_learning_rate_sweep(learning_rates, steps=40)
rate_diagnoses = {rate: diagnose_dynamics(trace) for rate, trace in rate_traces.items()}
for rate, trace in rate_traces.items():
    print(
        f"rate={rate:g} diagnosis={rate_diagnoses[rate]:24s} "
        f"final_loss={trace[-1].loss_after:.8g} "
        f"sign_changes={coordinate_sign_changes(trace, 1)}"
    )
assert rate_diagnoses == {
    1.0e-5: "stagnating",
    0.05: "converging",
    0.19: "oscillatory_convergence",
    0.21: "diverging",
}

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for rate, trace in rate_traces.items():
    ax.semilogy(loss_history(trace), label=f"lr={rate:g}: {rate_diagnoses[rate]}")
ax.set(xlabel="update", ylabel="loss (log scale)", title="One objective, four learning-rate regimes")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
plt.close(fig)

### Explain the observation

Use loss, parameter sign, parameter magnitude, and update magnitude together. A tiny positive change is stagnation over this fixed budget, not a zero gradient. Repeated crossings with shrinking magnitude indicate oscillatory convergence; crossings with growing magnitude indicate divergence.

## Predict before running — stability boundary

For the high-curvature coordinate, derive the exact-gradient multiplier `1 - learning_rate * 10`. Predict what changes immediately below (`0.19`) and above (`0.21`) the boundary at `0.20`. State the expected sign and magnitude trend before running.

In [ ]:
boundary_traces = run_learning_rate_sweep((0.19, 0.21), steps=12)
for rate, trace in boundary_traces.items():
    values = [trace[0].parameters_before[1]] + [record.parameters_after[1] for record in trace]
    print(f"lr={rate}: multiplier={1 - 10 * rate:+.2f}")
    print("  theta_1:", [round(value, 5) for value in values[:7]])
    magnitudes = [abs(value) for value in values]
    if rate < 0.20:
        assert all(after < before for before, after in zip(magnitudes, magnitudes[1:]))
    else:
        assert all(after > before for before, after in zip(magnitudes, magnitudes[1:]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for rate, trace in boundary_traces.items():
    values = [trace[0].parameters_before[1]] + [record.parameters_after[1] for record in trace]
    ax.plot(values, marker="o", label=f"lr={rate}")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set(xlabel="update", ylabel="high-curvature parameter", title="Decaying versus expanding oscillation")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
plt.close(fig)

## Predict before running — Controlled failure A: too-small-rate stagnation

Predict the relative loss improvement after 40 updates at `1e-5` and whether the gradient is absent, wrong, or merely scaled into tiny updates. Preserve objective, initialization, optimizer, and budget in your proposed repair.

In [ ]:
too_small_trace = run_optimizer("gd", learning_rate=1.0e-5, steps=40)
too_small_losses = loss_history(too_small_trace)
too_small_progress = (too_small_losses[0] - too_small_losses[-1]) / too_small_losses[0]
print(f"relative improvement: {too_small_progress:.4%}")
print("first gradient:", too_small_trace[0].gradient)
print("first update:", too_small_trace[0].applied_update)
assert too_small_progress < 0.01
assert too_small_trace[0].gradient == initial_gradient

### Diagnose before repair

The non-zero gradient is unchanged from the stable run. The isolated cause is update scale. Record why changing the objective, gradient, start, or budget would invalidate the repair comparison.

In [ ]:
small_rate_repair = run_optimizer("gd", learning_rate=0.05, steps=40)
print("failed final loss :", too_small_trace[-1].loss_after)
print("repaired final loss:", small_rate_repair[-1].loss_after)
assert small_rate_repair[0].parameters_before == too_small_trace[0].parameters_before
assert small_rate_repair[0].gradient == too_small_trace[0].gradient
assert small_rate_repair[-1].loss_after < too_small_trace[-1].loss_after

## Predict before running — Controlled failure B: too-large-rate divergence

For `learning_rate = 0.21`, predict the first four high-curvature parameter signs and whether their absolute values shrink or grow. Predict the loss direction. Name one observation that would contradict divergence.

In [ ]:
too_large_trace = run_optimizer("gd", learning_rate=0.21, steps=20)
too_large_values = [too_large_trace[0].parameters_before[1]] + [record.parameters_after[1] for record in too_large_trace]
print("theta_1 first values:", [round(value, 5) for value in too_large_values[:6]])
print("loss before/after:", too_large_trace[0].loss_before, too_large_trace[-1].loss_after)
assert coordinate_sign_changes(too_large_trace, 1) == 20
assert all(abs(after) > abs(before) for before, after in zip(too_large_values, too_large_values[1:]))
assert too_large_trace[-1].loss_after > too_large_trace[0].loss_before

### Diagnose before repair

Trace `gradient → optimizer state → update → parameters → next loss`. Exact GD has no hidden state, so the gradient remains correct while the learning rate crosses the high-curvature stability boundary. The smallest repair changes only that rate.

In [ ]:
large_rate_repair = run_optimizer("gd", learning_rate=0.19, steps=20)
assert large_rate_repair[0].parameters_before == too_large_trace[0].parameters_before
assert large_rate_repair[0].gradient == too_large_trace[0].gradient
assert large_rate_repair[-1].loss_after < large_rate_repair[0].loss_before
print("repaired diagnosis:", diagnose_dynamics(large_rate_repair))
print("repaired final loss:", large_rate_repair[-1].loss_after)

## Optimizer families on one objective

The objective and initialization remain fixed. GD uses the exact gradient. SGD uses exact gradient plus one seeded component perturbation whose full-fixture mean is zero. Momentum accumulates velocity. Adam accumulates bias-corrected first and second moments. Record optimizer-specific hyperparameters; the same numeric learning rate is not equally tuned by definition.

## Predict before running — GD versus seeded SGD

At the same start, rate, and update budget, predict which loss curve is smoother. Predict whether replaying one seed exactly reproduces the trace and whether a second seed changes component order. State why noisy individual updates can still target the same aggregate objective.

In [ ]:
gd_trace = run_optimizer("gd", learning_rate=0.05, steps=40)
sgd_trace = run_optimizer("sgd", learning_rate=0.05, steps=40, seed=2020)
print("component-noise mean:", component_noise_mean())
print("GD final loss :", gd_trace[-1].loss_after)
print("SGD final loss:", sgd_trace[-1].loss_after)
print("SGD sources   :", [record.gradient_source for record in sgd_trace[:8]])
assert component_noise_mean() == (0.0, 0.0)
assert all(record.gradient_source == "exact" for record in gd_trace)
assert all(record.gradient_source.startswith("component:") for record in sgd_trace)

In [ ]:
sgd_replay = run_optimizer("sgd", learning_rate=0.05, steps=40, seed=2020)
sgd_other_seed = run_optimizer("sgd", learning_rate=0.05, steps=40, seed=2021)
assert sgd_replay == sgd_trace
assert sgd_other_seed != sgd_trace
print("same-seed replay exact:", sgd_replay == sgd_trace)
print("different seed changes trace:", sgd_other_seed != sgd_trace)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(loss_history(gd_trace), label="GD exact gradient")
ax.semilogy(loss_history(sgd_trace), label="SGD seed 2020")
ax.semilogy(loss_history(sgd_other_seed), alpha=0.7, label="SGD seed 2021")
ax.set(xlabel="update", ylabel="loss (log scale)", title="Smooth and noisy paths on the same objective")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
plt.close(fig)

### Explain the observation

Separate reproducibility from smoothness. A seeded stochastic trace can be exactly reproducible while remaining noisy. A different seed is a sensitivity check, not permission to select the most favorable run.

## Predict before running — momentum trade-off

With `learning_rate = 0.05`, `momentum = 0.9`, and zero initial velocity, calculate the first two velocity vectors. Predict whether momentum beats GD after 5 updates and after 40 updates, and whether its loss stays monotonic. Record both horizon predictions before running.

In [ ]:
momentum_trace = run_optimizer("momentum", learning_rate=0.05, momentum=0.9, steps=40)
print("first direction :", momentum_trace[0].optimizer_direction)
print("second direction:", momentum_trace[1].optimizer_direction)
print("second parameters:", momentum_trace[1].parameters_after)
assert momentum_trace[0].optimizer_direction == (4.0, 40.0)
assert momentum_trace[1].optimizer_direction == (7.4, 56.0)

gd_5 = run_optimizer("gd", learning_rate=0.05, steps=5)
momentum_5 = run_optimizer("momentum", learning_rate=0.05, momentum=0.9, steps=5)
print("loss after 5 — GD/momentum:", gd_5[-1].loss_after, momentum_5[-1].loss_after)
print("loss after 40 — GD/momentum:", gd_trace[-1].loss_after, momentum_trace[-1].loss_after)
assert gd_5[-1].loss_after < momentum_5[-1].loss_after
assert momentum_trace[-1].loss_after < gd_trace[-1].loss_after

In [ ]:
momentum_losses = loss_history(momentum_trace)
momentum_increases = sum(after > before for before, after in zip(momentum_losses, momentum_losses[1:]))
print("momentum loss-increase steps:", momentum_increases)
print("high-curvature sign changes:", coordinate_sign_changes(momentum_trace, 1))
assert momentum_increases > 0
assert coordinate_sign_changes(momentum_trace, 1) > 0

### Momentum is a trade-off, not a universal win

The ranking changes with horizon: velocity can accelerate progress and can also carry a parameter past a low-loss region. Report overshoot and non-monotonicity alongside final loss.

## Predict before running — Adam trade-off

Adam normalizes each first-step coordinate by its estimated scale. Predict the first optimizer direction from gradients `(4, 40)`. Then predict whether Adam at rate `0.05` is automatically better than GD at `0.05`, and how the comparison may change after tuning Adam to `0.30` for this fixture.

In [ ]:
adam_same_rate = run_optimizer("adam", learning_rate=0.05, steps=40)
adam_tuned_fixture = run_optimizer("adam", learning_rate=0.30, steps=40)
print("Adam first direction:", adam_same_rate[0].optimizer_direction)
print("Adam first update   :", adam_same_rate[0].applied_update)
print("final loss Adam lr=.05:", adam_same_rate[-1].loss_after)
print("final loss Adam lr=.30:", adam_tuned_fixture[-1].loss_after)
print("final loss GD   lr=.05:", gd_trace[-1].loss_after)
assert all(abs(value - 1.0) < 1.0e-8 for value in adam_same_rate[0].optimizer_direction)
assert adam_same_rate[-1].loss_after > gd_trace[-1].loss_after
assert adam_tuned_fixture[-1].loss_after < gd_trace[-1].loss_after

In [ ]:
comparison_runs = [
    ("GD", 0.05, gd_trace),
    ("SGD seed 2020", 0.05, sgd_trace),
    ("momentum beta=.9", 0.05, momentum_trace),
    ("Adam same rate", 0.05, adam_same_rate),
    ("Adam fixture-tuned", 0.30, adam_tuned_fixture),
]
print(f"{'optimizer':22s} {'lr':>7s} {'final loss':>14s} {'loss rises':>12s}")
for name, rate, trace in comparison_runs:
    losses = loss_history(trace)
    rises = sum(after > before for before, after in zip(losses, losses[1:]))
    print(f"{name:22s} {rate:7.3f} {losses[-1]:14.8f} {rises:12d}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, rate, trace in comparison_runs:
    ax.semilogy(loss_history(trace), label=f"{name}, lr={rate:g}")
ax.set(xlabel="update", ylabel="loss (log scale)", title="Optimizer evidence depends on tuning and horizon")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.show()
plt.close(fig)

## Reject universal superiority

These traces justify only fixture-bounded claims. A fair report names the objective, initialization, update budget, learning rate, optimizer state hyperparameters, stochastic seed, and metric. Adam's tuned fixture run does not prove Adam is universally superior; GD's strong five-step result does not prove momentum is inferior.

## Code reading — gradient → optimizer state → applied update → parameters → next loss

Read `run_optimizer` in `missions/M20/optimization_core.py`. Before printing the records below, trace the first step of GD, momentum, and Adam by hand. Identify which direction equals the gradient and which directions incorporate state or scale adaptation.

In [ ]:
for trace in (gd_trace, momentum_trace, adam_same_rate):
    record = trace[0]
    print("\noptimizer:", record.optimizer)
    print("parameters before :", record.parameters_before)
    print("gradient          :", record.gradient)
    print("optimizer direction:", record.optimizer_direction)
    print("applied update    :", record.applied_update)
    print("parameters after  :", record.parameters_after)
    print("next loss         :", record.loss_after)

## Evidence contract

Submit timestamped predictions, complete traces, explanations of surprises, preserved controls, all optimizer hyperparameters, stochastic seeds and sensitivity, both failure repairs, and fixture-bounded trade-off claims. Do not substitute these printed reference values for learner-produced evidence.

## No-AI gate

Close this notebook and complete `missions/M20/no_ai_gate.md` from a blank page without AI-generated code, calculations, prose, or diagrams. The repository intentionally contains no answers, score, completion claim, or signed evidence.

## Unfilled ADR and formal review

Use `missions/M20/adr_prompt.md` to author an optimizer and learning-rate policy. Decision, evidence, alternatives, trade-offs, monitoring, rollback/revisit triggers, owner, status, and date remain **UNFILLED BY LEARNER**. Reviewers use `review_brief.md`; implementation completion is not learner sign-off.

## M19 → M20 → M21 handoff

M19 explains gradients. M20 manipulates learning dynamics and closes V04's Mathematical Instrumentation Layer. M21 may begin black-box neural-network training only with recorded hyperparameters, seed policy, monitored loss/update signals, divergence and stagnation triggers, rollback behavior, and an approved learner ADR.

## Mission summary prompt

In your own words, explain why one learning rate stagnated, why rates around the stability boundary oscillated differently, how SGD remained reproducible, what momentum and Adam changed after the gradient, and why no trace establishes a universally best optimizer. Do not prefill this explanation in source control.